In [1]:
import os
os.environ["LOG_LEVEL"]="WARNING"

## 1. Configure Nodes

In [2]:
import os
from gai.dialogue import FileDialogueBus
from gai.lib.config.gai_client_config import GaiClientConfig
from gai.lib.tests import make_local_tmp

dialogue_id = FileDialogueBus.create_dialogue_id()
here = make_local_tmp()
dialogue = FileDialogueBus(
    logger_name="User",
    app_dir=here,
    dialogue_id=dialogue_id,
    reset=True
)
await dialogue.start()

from gai.mace.nodes import MaceUserNode, MaceAgentNode
user_node = MaceUserNode(dialogue=dialogue)
await user_node.register()

from gai.mace import Persona
llm_config = GaiClientConfig(**{
    "client_type": "anthropic",
    "model": "claude-opus-4-20250514",
})
sara = MaceAgentNode(
    dialogue=dialogue, 
    persona=Persona(
        name="Sara",
        sex="F",
        job_description="You are a helpful assistant ready to answer any questions from the user",
        self_introduction="Hi! My name is Sara! I am always at your service. Just let me know what you need.",
        skills="conversation",
        traits="kind,empathy,diligent",
        agent_class="ChatAgent",
        dialogue=dialogue,
        llm_config=llm_config
))
await sara.register()
diana = MaceAgentNode(
    dialogue=dialogue, 
    persona=Persona(
        name="Diana",
        sex="F",
        job_description="You are a dissenter and will always offer a different opinion from the others.",
        self_introduction="Hi! My name is Diana! You can always count on me to give you a different opinion.",
        skills="conversation",
        traits="meticulous,scrutiny,diligent",
        agent_class="ChatAgent",
        dialogue=dialogue,
        llm_config=llm_config
))
await diana.register()

---

## 2. Test Rollcall

In [3]:
from IPython.display import display, HTML

participants = await user_node.rollcall()

html = ""
for participant in participants:
    html += f"""
    <div style="display: flex; align-items: center; margin-bottom: 1.5em; border: 1px solid #ddd; padding: 10px; border-radius: 8px; max-width: 500px;">
        <div style="flex: 1;">
            <div><strong>Agent:</strong> {participant.name}</div>
            <div><strong>Class:</strong> {participant.agent_class}</div>
            <div><strong>Description:<br></strong> {participant.desc}</div>
        </div>
        <div style="margin-left: 20px;align-self: flex-start;">
            <img src="{participant.image_128x128}" style="width:128px; height:128px; border-radius: 4px;" />
        </div>
    </div>
    """
display(HTML(html))

---

## 3. Test Chaining

In [4]:
flow = """
    User ->> Sara
    Sara ->> Diana
    """
stream =  user_node.chat(
    flow=flow,
    round_no=0,
    user_message="Tell me a one paragraph story",    
    )
start=True
async for chunk in stream:
    if start:
        print(chunk.header.sender+":")
        start=False
    print(chunk.body.chunk,end="",flush=True)


Sara:
The old lighthouse keeper noticed something peculiar that storm-swept evening—the waves were climbing the rocks in reverse, flowing up and away from the shore like liquid mercury defying gravity. As he pressed his weathered face against the tower window, the ocean began to peel away from the earth entirely, revealing an ancient city of gleaming coral spires that had slumbered beneath the waters for millennia. The city's bells began to ring, though no hand pulled their ropes, and from the barnacle-crusted streets emerged his daughter Maya, who had vanished in a shipwreck twenty years ago, her eyes now the color of deep tide pools and her hair flowing like kelp in an invisible current. She raised her hand to wave, smiling that same crooked smile he remembered, and he understood in that moment that the sea had been keeping her safe all along, waiting for the night when the veils between worlds grew thin enough for her to finally come home.

In [5]:
start=True
async for chunk in user_node.next():
    if start:
        print(chunk.header.sender+":")
        start=False
    print(chunk.body.chunk,end="",flush=True)

Diana:
That was beautiful! The image of the waves flowing in reverse is so haunting and dreamlike. I love how you wove together the supernatural elements with such deep emotion - the father's loss and the bittersweet revelation that his daughter has been transformed but preserved by the sea. The detail about her eyes being "the color of deep tide pools" really drives home her otherworldly transformation while keeping her recognizably his daughter. 

Could you tell me another short story, but this time make it about someone who discovers they can hear the thoughts of inanimate objects?